Finding the best site the bet on is a time-consuming and challenging task. With odds constantly changing, finding what place is giving you the best odds manually is nearly impossible. Initially, our objective was to create a web-scraping algorithm to scrape odds from different betting sites in order to obtain our information. We quickly realized that this process would be too difficult since 1. betting sites encrypt their data when you inspect their page to prevent web scraping and 2. if we were to manage to webscrape a betting site, many betting sites update their encryptions periodically which could prove to be another problem down the road. 

Instead, we found a sports betting API (the-odds-api) that compiles the moneyline and spread odds from multiple sports betting sites. Due to the limited number of games per day, we want to create a pipeline that cleans the data we retrieve as a json into a more readable nested dictionary then as a pandas dataframe. Through this process we also want to split the data, so that we have different dataframes for different markets (head to head, spreads, and totals), which we will then use for our visualizations and analysis. 

As for our machine learning aspect of the project, we still want to focus on clustering. Since classification is more for estimating a class based on a few parameters and regression is used for estimating another parameters given other parameters, we feel like clustering based on price, odds, or expected return of a certain betting site will be able to create usable groups where we can differentiate and choose which site and which format of bets are the best.

For the two visualizations we made for the Data and Analysis Plan, we decided on a bar graph comparing the odds for an individual game by betting site and a pie chart representing how often each betting site had the lowest odds for any game in the dataframe. As of right now, our data is only the games on 4/1 because we haven't been adding games, but in the future we plan to continue to append more games as the semester goes on.

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json

#Source: https://the-odds-api.com/
url = f'https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?apiKey=eaa265db381b34b1dbe2ce35e2628048&regions=us&markets=h2h,spreads&oddsFormat=american'
betting_text = requests.get(url).text
betting_data = json.loads(betting_text)

In [2]:
# Length of the betting_data
len(betting_data)

8

In [3]:
# looking betting data
sites_list = list()
for game in betting_data:
    bookmakers = game['bookmakers']
    for site in bookmakers:
        sites_list.append(site['title'])

In [4]:
sites_list = list(set(sites_list))

In [5]:
#init a dictionary so that we get our first look at the data
data_by_game = dict()



for game in betting_data:
    # Get the home team and away team by game and format it for easier viewing
    
    home = game['home_team']
    away = game['away_team']
    
    matchup = f'{away} @ {home}'
    
    # Keep the matchup name as the key, will store odds by site as value
    data_by_game[matchup] = dict()
    
    bookmakers = game['bookmakers']
    for site in bookmakers:
        site_title = site['title']
        data_by_game[matchup][site_title] = dict()
        markets = site['markets']
        
        for market in markets:
            bet_format = market['key']
            data_by_game[matchup][site_title][bet_format] = market['outcomes']
                     
#import pretty printer
import pprint

#this library helps us see all of the nested dictionaries 
pp = pprint.PrettyPrinter(indent=4)
pp.pprint(data_by_game)

{   'Atlanta Hawks @ Cleveland Cavaliers': {   'Barstool Sportsbook': {   'h2h': [   {   'name': 'Atlanta '
                                                                                                 'Hawks',
                                                                                         'price': -125},
                                                                                     {   'name': 'Cleveland '
                                                                                                 'Cavaliers',
                                                                                         'price': 106}],
                                                                          'spreads': [   {   'name': 'Atlanta '
                                                                                                     'Hawks',
                                                                                             'point': -2.0,
                          

                                                                                  {   'name': 'Golden '
                                                                                              'State '
                                                                                              'Warriors',
                                                                                      'point': -6.0,
                                                                                      'price': -110}]},
                                                  'BetUS': {   'h2h': [   {   'name': 'Denver '
                                                                                      'Nuggets',
                                                                              'price': 210},
                                                                          {   'name': 'Golden '
                                                                                      'State '
     

                                                                                {   'name': 'Toronto '
                                                                                            'Raptors',
                                                                                    'point': 4.5,
                                                                                    'price': -110}]},
                                                'LowVig.ag': {   'h2h': [   {   'name': 'Philadelphia '
                                                                                        '76ers',
                                                                                'price': -180},
                                                                            {   'name': 'Toronto '
                                                                                        'Raptors',
                                                                                'price': 160}],
   

## Build Data DataFrame and Visualisations 

In [7]:

# init dataframe
df_bets = pd.DataFrame()

for game in betting_data:
    # Get the home team and away team by game and format it for easier viewing
    
    team_home = game['home_team']
    team_away = game['away_team']
    Game_ID = game['id']
    
    bookmakers = game['bookmakers']
    
    for site in bookmakers:
        site_title = site['title']
        #data_by_game[matchup][site_title] = dict()
        markets = site['markets']
        
             
        for market in markets:
            bet_format = market['key']
            bet = market['outcomes']
            
            
            # create a dictionary and add all of the required items to it 
            bet_dict = {}
            bet_dict['Game_ID'] = Game_ID
            bet_dict['bet_format'] = market['key']
            bet_dict['site_title'] = site_title
            bet_dict['home_team'] = team_home
            bet_dict['away_team'] = team_away
            
            bet_dict['name_away'] = bet[1]['name']
            bet_dict['price_away'] = bet[1]['price']
            bet_dict['name_home'] = bet[0]['name']
            bet_dict['price_home'] = bet[0]['price']
            
            df_bets = df_bets.append(bet_dict, ignore_index = True)
     

    
df_bets.to_csv('15_April')

In [ ]:
df_bets_grouped = df_bets.groupby('Game_ID')

In [ ]:
# bool index to head to head bets 
h2h = df_bets['bet_format'] == 'h2h'
df_bets_h2h = df_bets.loc[h2h, :]


#Now for set game ID as inidexing to try and simply the viz 
df_bets_h2h = df_bets_h2h.set_index('Game_ID')

# lets look at the first game 
game_1 = df_bets_h2h.loc[df_bets_h2h.index[0]]


#Import some additional packages
import matplotlib.pyplot as plt
import seaborn as sns


# Building a vizualisaiton best for the 10 options from our dataset to show that there is a significant difference
sns.set()
chart = sns.barplot(x="site_title",y = "price_away",data= game_1.iloc[:10])
chart.set_xticklabels(chart.get_xticklabels(), rotation=45)
chart.set_xlabel("Bookie", fontsize = 20)
chart.set_ylabel("Price for Home Team bet ", fontsize = 20)

home_team_Str = game_1['home_team'][0]
away_team_Str = game_1['away_team'][0]

chart.set_title(f'{home_team_Str} @ {away_team_Str} ', fontsize = 20)

In [ ]:
## Very similar to the one abovea

game_2 = df_bets_h2h.loc[df_bets_h2h.index[44]]

import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
chart = sns.barplot(x="site_title",y = "price_away",data= game_2.iloc[:10])
chart.set_xticklabels(chart.get_xticklabels(), rotation=45)
chart.set_xlabel("Bookie", fontsize = 20)
chart.set_ylabel("Price for Home Team bet ", fontsize = 20)

home_team_Str = game_2['home_team'][0]
away_team_Str = game_2['away_team'][0]

chart.set_title(f'{away_team_Str} @ {home_team_Str}', fontsize = 20)

In [ ]:
df_bets

In [ ]:
df_bets_ = df_bets.set_index('Game_ID','site_title')

# Get just the h2h odds
h2h = df_bets_['bet_format'] == 'h2h'
df_bets_h2h = df_bets_.loc[h2h, :]

# Select what columns we want to keep to make the dataframe easier to work with
df_bets_h2h = df_bets_h2h[['bet_format', 'away_team', 'home_team', 'price_home', 'site_title']]
min_home_odds = df_bets_h2h.groupby('home_team')['price_home'].min()

min_odds_dict = min_home_odds.to_dict()


In [ ]:
min_site_counts = list()


for key, value in min_odds_dict.items():
    row = df_bets_h2h.loc[(df_bets_h2h['home_team'] == key) & (df_bets_h2h['price_home'] == value)]
    site = row['site_title']
    min_site_counts += list(site.values)

min_site_counts

In [ ]:
from collections import Counter

site_counts = dict(Counter(min_site_counts))

# Initialize lists for pie chart
labels = list()
sizes = list()

# Split up dictionary into separate lists
for site, count in site_counts.items():
    labels.append(site)
    sizes.append(count)
    
plt.pie(sizes, labels = labels, autopct='%1.1f%%')
plt.axis('equal')
plt.title('Percentage of Best Odds by Site')
plt.show()